<a href="https://colab.research.google.com/github/alexfremier/FunctionalIntegrity/blob/main/IntegrityTool_Phase2_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Functional Integrity — Phase 2 (v2, patched)

**Service-shed functional integrity from Dynamic World (10 m) and ESA CCI (300 m)**

Rebuilt from `IntegrityTool_Phase2_skeleton.ipynb` after the 2026-07-17 Analysis 1 run
failed with `User memory limit exceeded` (AUS, DW10) and `Computation timed out` (DW300).

**What changed and why**

| # | Change | Fixes |
|---|--------|-------|
| 1 | `dw_annual_label` uses band-wise max instead of `toArray().arrayArgmax()` | DW300 timeout |
| 2 | `service_shed_pct` uses a single annulus kernel (2 neighborhood passes, not 4) | DW10 memory |
| 3 | Per-country `filterBounds` instead of the global subset bbox | wasted recomputation |
| 4 | `ee.Image(0)` anchored to the CCI projection in `cci_integrity` | projection failures |
| 5 | `tileScale`, explicit `crs`, and per-country tasks in `process`/`export_table` | memory headroom |
| 6 | `cancel_tasks` helper uses `cancelOperation` | `cancelTask` silently no-ops on this API |

**Run order:** cells 1-7 define everything (no compute). Then run the *smoke test*
(section 8) on 3 small countries before launching anything larger. Section 9 is the
staged submission harness with throttling.


## 1 · Setup & auth

In [ ]:
import ee, math, time
from collections import Counter
from datetime import datetime, timezone

PROJECT = 'ee-fremier'
ee.Authenticate()
ee.Initialize(project=PROJECT)
print('initialized:', PROJECT)

## 2 · Configuration

In [ ]:
# ---- Boundaries --------------------------------------------------------------
GADM_ASSET = "projects/ee-fremier/assets/GADM_GID_0_simplified"
ID_FIELD   = "GID_0"

# ---- Datasets ----------------------------------------------------------------
CCI_COLLECTION = "projects/sat-io/open-datasets/ESA/C3S-LC-L4-LCCS"   # 300 m, band b1
CCI_BAND       = "b1"
DW_COLLECTION  = "GOOGLE/DYNAMICWORLD/V1"                             # 10 m, per-scene
DW_CLASSES = ['water','trees','grass','flooded_vegetation','crops',
              'shrub_and_scrub','built','bare','snow_and_ice']        # label 0..8

# ---- Service-shed geometry (METRES, from focal-cell CENTRE) -------------------
FOCAL_HALF_M    = 150
SERVICE_REACH_M = 1000
INNER_RADIUS_M  = FOCAL_HALF_M                     # 150 m
OUTER_RADIUS_M  = FOCAL_HALF_M + SERVICE_REACH_M   # 1150 m
INNER_RADIUS_M_10M = 5                             # Analysis 3 only

# ---- Analysis parameters -----------------------------------------------------
THRESHOLD_PCT = 20
ANCHOR_YEAR   = 2021
SCALE_300     = 300
SCALE_10      = 10
CRS           = 'EPSG:4326'      # pinned everywhere; never inherited

# ---- Memory / concurrency knobs ---------------------------------------------
TILESCALE_300 = 4
TILESCALE_10  = 16               # giants need the headroom
MAX_INFLIGHT  = 3                # Contributor tier: keep the queue shallow

# ---- Stratified subset (Analysis 1) -----------------------------------------
SUBSET = ['NGA','IND','THA','UKR','FRA','CAN','AUS','ETH','ZAF','CHN','IDN',
          'USA','MEX','BRA','ZMB','MDG','MYS','PNG','PRY','CHL','PER']

# Smallest-first ordering: fail fast and cheap before committing to the giants.
SMOKE_TEST = ['PRY','ZMB','PNG']
GIANTS     = ['RUS','CAN','USA','CHN','BRA','AUS','IND','ZMB','PNG']   # need tileScale=16
# Countries reduced by GADM sub-unit instead of whole (they time out whole at 10 m):
SPLIT      = {'ZMB','PNG'}

## 3 · Service-shed kernel (single annulus)

The original built the ring as *outer circle minus inner circle*, which cost **four**
`reduceNeighborhood` passes. At 10 m the outer radius is 115 pixels (~41,000 px per
kernel), so four passes is what exceeded the per-task memory ceiling on Australia.

Building the annulus directly as one fixed kernel halves that to **two** passes
(one for the ring sum, one for the valid-pixel count) with identical semantics.

The kernel is now grain-aware, so `scale_m` must be passed in.

In [ ]:
_KERNEL_CACHE = {}

def annulus_kernel(inner_m, outer_m, scale_m):
    """Fixed kernel: 1.0 strictly outside inner radius, out to and including outer."""
    key = (inner_m, outer_m, scale_m)
    if key in _KERNEL_CACHE:
        return _KERNEL_CACHE[key]

    r_out = int(round(outer_m / scale_m))
    r_in  = int(round(inner_m / scale_m))
    if r_out > 200:
        raise ValueError(
            f"annulus radius {r_out} px is very large ({2*r_out+1}^2 kernel). "
            "Consider pre-aggregating `natural` (see section 6) before this step.")

    w = []
    for dy in range(-r_out, r_out + 1):
        row = []
        for dx in range(-r_out, r_out + 1):
            d = math.hypot(dx, dy)
            row.append(1.0 if (r_in < d <= r_out) else 0.0)
        w.append(row)

    k = ee.Kernel.fixed(weights=w, normalize=False)
    _KERNEL_CACHE[key] = k
    return k


def service_shed_pct(natural, inner_m, outer_m, scale_m):
    """Percent-natural over the annulus around each pixel.

    `natural` is the integrity-weight image, masked where water/ice.
    ring mean = (sum natural in ring) / (count valid in ring) * 100
    """
    k = annulus_kernel(inner_m, outer_m, scale_m)
    sum_r = ee.Reducer.sum().unweighted()

    valid    = natural.mask()
    ring_sum = natural.reduceNeighborhood(sum_r, k)
    ring_cnt = valid.reduceNeighborhood(sum_r, k)
    ring_cnt = ring_cnt.where(ring_cnt.lte(0), 1)          # guard div-by-zero

    return ring_sum.divide(ring_cnt).multiply(100).rename('shed_pct')

## 4 · Integrity / ag crosswalks

Unchanged in substance. One fix: the naturalness surface was built from `ee.Image(0)`,
a constant with **no fixed projection** — the most likely source of the Phase 1
projection failures. It is now anchored to the CCI projection.

In [ ]:
def cci_integrity(lc, harmonized=False):
    """lc = CCI LCCS class image (band b1)."""
    base = lc.multiply(0)                      # anchored to lc's projection (was ee.Image(0))

    nat = (base
           .where(lc.eq(30), 0.25 if not harmonized else 0.0)   # mosaic cropland
           .where(lc.eq(40), 0.75 if not harmonized else 1.0)   # mosaic natural
           .where(lc.gte(50).And(lc.lte(180)), 1.0)             # natural veg
           .where(lc.gte(200).And(lc.lte(202)), 1.0))           # bare
    nat = nat.updateMask(lc.neq(210).And(lc.neq(220)))          # mask water / ice

    if harmonized:
        ag = lc.gte(10).And(lc.lte(30))        # 40 -> natural
    else:
        ag = lc.gte(10).And(lc.lte(40))        # Phase 1 scheme
    return nat.rename('natural'), ag.rename('ag')


def dw_integrity(label):
    """label = DW annual class 0..8 (see DW_CLASSES)."""
    # water0 / ice8 -> -1 (masked); crops4 / built6 -> 0; trees1 grass2 flooded3 shrub5 bare7 -> 1
    nat = label.remap([0,1,2,3,4,5,6,7,8],
                      [-1,1,1,1,0,1,0,1,-1]).rename('natural')
    nat = nat.updateMask(nat.gte(0))
    ag  = label.eq(4).rename('ag')
    return nat, ag

## 5 · Dynamic World annual composite

**This is what timed out.** The original computed the annual label via
`mean_prob.toArray().arrayArgmax().arrayGet(0)`. Array operations are the most
memory-hungry primitive in Earth Engine, and this ran per output pixel across a
near-global extent.

The replacement takes the max across the 9 probability bands, then finds which band
equals it — identical argmax semantics, no array conversion.

Tie-breaking: this resolves ties to the **highest** class index. `arrayArgmax` resolves
to the lowest. Ties in a mean-probability surface are vanishingly rare; set
`ties='low'` to match the original exactly.

In [ ]:
def dw_annual_label(year, region, ties='high'):
    """Calendar-year DW label = argmax of the mean per-class probability."""
    start, end = f'{year}-01-01', f'{year+1}-01-01'
    coll = (ee.ImageCollection(DW_COLLECTION)
              .filterDate(start, end)
              .filterBounds(region)            # per-country, NOT the global subset bbox
              .select(DW_CLASSES))

    mean_prob = coll.mean()
    max_prob  = mean_prob.reduce(ee.Reducer.max())

    order = range(1, 9) if ties == 'high' else range(7, -1, -1)
    label = ee.Image(8 if ties == 'low' else 0)
    for i in order:
        label = label.where(mean_prob.select(i).eq(max_prob), i)

    return label.rename('label').toInt().updateMask(max_prob.gt(0))

## 6 · Core processing

Adds `tileScale` and an explicit `crs`, and takes `scale_m` through to the kernel.

`bestEffort` is deliberately **not** enabled: with it on, Earth Engine silently coarsens
the scale to make a reduction fit, which would leave different countries computed at
different effective resolutions. A loud failure is better than silent heterogeneity.

`preaggregate_m` is the escape hatch for countries that still exceed memory at 10 m: it
averages `natural` up to a coarser grain first (preserving fractional naturalness), so the
kernel radius in pixels shrinks quadratically. Ring means stay very close, but this is an
approximation and must be reported if used.

In [ ]:
def process(natural, ag, inner_m, outer_m, scale, regions, year, tag,
            tile_scale=4, preaggregate_m=None):
    """-> FeatureCollection with functional integrity fraction + ag area."""

    kernel_scale = scale
    if preaggregate_m:
        factor  = int(preaggregate_m // scale)
        natural = (natural
                   .setDefaultProjection(crs=CRS, scale=scale)   # FIX: reduceResolution needs a projection
                   .reduceResolution(ee.Reducer.mean(), maxPixels=factor * factor * 4)
                   .reproject(crs=CRS, scale=preaggregate_m))
        kernel_scale = preaggregate_m

    shed  = service_shed_pct(natural, inner_m, outer_m, kernel_scale)
    integ = shed.gte(THRESHOLD_PCT).updateMask(ag).rename('fi')
    ag_ha = ee.Image.pixelArea().updateMask(ag).divide(1e4).rename('agArea_ha')
    img   = integ.addBands(ag_ha)

    reducer = (ee.Reducer.mean().setOutputs(['fi'])
               .combine(ee.Reducer.sum().unweighted().setOutputs(['agArea_ha']),
                        sharedInputs=False))

    fc = img.reduceRegions(collection=regions,
                           reducer=reducer,
                           scale=scale,
                           crs=CRS,
                           tileScale=tile_scale)

    return fc.map(lambda f: f.set({'year': year, 'product': tag}))

## 7 · Task helpers

`ee.data.cancelTask()` **silently no-ops** on the Cloud-API-backed client — tasks are
Operations addressed by resource name, not by bare ID. The 2026-07-17 debugging session
lost ~16 hours to this. Always use `cancelOperation(t['name'])`.

In [ ]:
def list_tasks(prefix=None, states=None):
    ts = ee.data.getTaskList()
    if prefix:
        ts = [t for t in ts if t.get('description','').startswith(prefix)]
    if states:
        ts = [t for t in ts if t.get('state') in states]
    return ts


def task_summary(prefix=None):
    ts = list_tasks(prefix)
    print(Counter(t['state'] for t in ts))
    for t in ts:
        if t['state'] in {'READY','RUNNING'}:
            ms = t.get('update_timestamp_ms') or t.get('creation_timestamp_ms')
            when = (datetime.fromtimestamp(ms/1000, timezone.utc).strftime('%m-%d %H:%M')
                    if ms else '-')
            print(f"  {t['state']:8} {when}  {t['description']}")
        elif t['state'] == 'FAILED':
            print(f"  FAILED   {t['description']}\n    -> {t.get('error_message')}")


def cancel_tasks(prefix, dry_run=True):
    """Cancel READY/RUNNING tasks matching prefix. Uses cancelOperation."""
    targets = list_tasks(prefix, states={'READY','RUNNING'})
    print(f"{len(targets)} target(s)")
    for t in targets:
        print(f"  {t['state']:8} {t['description']}")
    if dry_run:
        print("\nDRY RUN - nothing cancelled. Pass dry_run=False to execute.")
        return
    print()
    for t in targets:
        try:
            ee.data.cancelOperation(t.get('name') or t['id'])
            print(f"  cancelled  {t['description']}")
        except Exception as e:
            print(f"  ERROR      {t['description']} -> {type(e).__name__}: {e}")


def export_table(fc, name, folder="EE_Integrity_Phase2"):
    t = ee.batch.Export.table.toDrive(
        collection=fc, description=name, folder=folder,
        fileNamePrefix=name, fileFormat="CSV",
        selectors=[ID_FIELD, "year", "product", "fi", "agArea_ha"])
    t.start()
    return t

## 8 · Smoke test — run this first

Three small countries at DW-10. Under the Contributor tier this should resolve in
well under an hour. If it does, the reducer configuration is sound and you can scale up.
If it fails, the problem is upstream in the composite, not in the reduction — and you
have found that out in minutes rather than overnight.

In [ ]:
def country_fc(gid):
    return ee.FeatureCollection(GADM_ASSET).filter(ee.Filter.eq(ID_FIELD, gid))


def build_dw10(gid, year=ANCHOR_YEAR, preaggregate_m=None):
    """One country, DW at native 10 m, comparison shed (inner 150 m / outer 1150 m)."""
    fc  = country_fc(gid)
    geo = fc.geometry()

    label       = dw_annual_label(year, geo)          # per-country bounds
    nat, ag     = dw_integrity(label)
    tile_scale  = TILESCALE_10 if gid in GIANTS else 8

    return process(nat, ag, INNER_RADIUS_M, OUTER_RADIUS_M, SCALE_10,
                   fc, year, 'DW10',
                   tile_scale=tile_scale, preaggregate_m=preaggregate_m)


# --- launch the smoke test ---------------------------------------------------
# smoke = []
# for gid in SMOKE_TEST:
#     t = export_table(build_dw10(gid), f"p2v2_DW10_{gid}_{ANCHOR_YEAR}")
#     smoke.append(t)
#     print('started', gid)

# task_summary('p2v2_')

## 8b · Sub-national split for timeout countries

Large, low-density countries (Zambia, Papua New Guinea) time out at 10 m even after
`tileScale` escalation and 30 m preaggregation, because focal-kernel cost scales with
**bounding-box extent**, not land area. The fix is to reduce each country over its GADM
sub-units (`GID_1`) instead of as one polygon.

`reduceRegions` processes the whole sub-unit collection in a **single task**, so splitting
Zambia into 115 units produces one export task with 115 rows — not 115 tasks. Each unit's
kernel covers a small bounding box, staying under the timeout, while queue load is unchanged.

`build_dw10_split` is identical to `build_dw10` except it passes the country's sub-units to
`process()` as `regions`; `process()` itself is untouched. Every row carries its own `fi`
and `agArea_ha` plus the parent country (`GID_0`) for rollup. Sub-unit boundaries come from
`GADM_GID_1`. Recombine with the rollup in section 8c.

In [ ]:
# --- sub-unit boundaries (uploaded GADM level-1/2 asset) --------------------
ADM1_ASSET = "projects/ee-fremier/assets/gadm_410_L1_simplified"
ADM1_C     = "GID_0"     # parent-country id, links sub-units to a country


def subunits_fc(gid):
    """All GADM sub-units belonging to country `gid`."""
    return ee.FeatureCollection(ADM1_ASSET).filter(ee.Filter.eq(ADM1_C, gid))


def build_dw10_split(gid, year=ANCHOR_YEAR, preaggregate_m=None):
    """Like build_dw10, but reduces over the country's GADM sub-units.
    One reduceRegions task; one output row per sub-unit. Recombine with
    combine_all() in section 8c."""
    regions = subunits_fc(gid)
    geo     = regions.geometry()

    label      = dw_annual_label(year, geo)
    nat, ag    = dw_integrity(label)
    tile_scale = TILESCALE_10 if gid in GIANTS else 8

    fc = process(nat, ag, INNER_RADIUS_M, OUTER_RADIUS_M, SCALE_10,
                 regions, year, 'DW10',
                 tile_scale=tile_scale, preaggregate_m=preaggregate_m)
    return fc.map(lambda f: f.set('GID_0', gid))   # tag parent country for rollup


def export_table_split(fc, name, folder="EE_Integrity_Phase2"):
    """Like export_table, but keeps the sub-unit id and parent country."""
    t = ee.batch.Export.table.toDrive(
        collection=fc, description=name, folder=folder,
        fileNamePrefix=name, fileFormat="CSV",
        selectors=[ADM1_C, "GID_1", "year", "product", "fi", "agArea_ha"])
    t.start()
    return t


def build_dw10_auto(gid, year=ANCHOR_YEAR, preaggregate_m=None):
    """Route split countries through build_dw10_split, others through build_dw10."""
    if gid in SPLIT:
        return build_dw10_split(gid, year, preaggregate_m)
    return build_dw10(gid, year, preaggregate_m)


# --- split smoke test ------------------------------------------------------
# split = []
# for gid in ['ZMB','PNG']:
#     t = export_table_split(build_dw10_split(gid, preaggregate_m=30),
#                            f"p2v2_DW10split_{gid}_{ANCHOR_YEAR}")
#     split.append(t); print('started split', gid)
# task_summary('p2v2_DW10split_')

## 8c · Rollup — area-weight sub-units back to national values

After the split export CSVs land in Drive, recombine each country's sub-units into one
national value:

    country_fi = Σ(fi_i · agArea_ha_i) / Σ(agArea_ha_i)

The weighting is **required**: `fi` is a fraction of agricultural land, so a plain mean of
sub-unit fractions would over-weight small units. Sub-units with no agricultural land
(`agArea_ha = 0`) carry zero weight and are dropped. Output matches the whole-country
schema plus a sub-unit count for QA. Plain pandas — needs only the mounted Drive, no active
EE session.

In [ ]:
import pandas as pd, glob, os

SPLIT_DIR = "/content/drive/MyDrive/EE_Integrity_Phase2"   # where the CSVs land


def combine_one(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df["agArea_ha"].fillna(0) > 0].copy()      # drop zero-ag units
    if df.empty:
        return None
    w  = df["agArea_ha"]
    fi = (df["fi"] * w).sum() / w.sum()                # area-weighted mean
    return {"GID_0": df["GID_0"].iloc[0], "year": df["year"].iloc[0],
            "product": df["product"].iloc[0], "fi": fi,
            "agArea_ha": w.sum(), "n_subunits": len(df)}


def combine_all(pattern="p2v2_DW10split_*.csv"):
    rows = []
    for path in sorted(glob.glob(os.path.join(SPLIT_DIR, pattern))):
        r = combine_one(path)
        if r:
            rows.append(r)
            print(f"  {r['GID_0']}: fi={r['fi']:.4f}  "
                  f"agArea_ha={r['agArea_ha']:,.0f}  ({r['n_subunits']} sub-units)")
    return pd.DataFrame(rows, columns=["GID_0","year","product","fi","agArea_ha","n_subunits"])


# national = combine_all()
# national.to_csv(os.path.join(SPLIT_DIR, "p2v2_DW10_national_from_split.csv"), index=False)
# national

## 9 · Staged submission (throttled)

The 2026-07-17 run submitted 20 tasks at once against a default allocation of roughly
2 average concurrent batch tasks. Nothing entered RUNNING for 16 hours.

This harness keeps at most `MAX_INFLIGHT` tasks live and feeds the queue as slots free,
smallest countries first, so a systematic failure surfaces early and cheaply.

In [ ]:
def submit_throttled(gids, build_fn, prefix, max_inflight=MAX_INFLIGHT, poll_s=120):
    """Submit one task per country, keeping the in-flight count bounded."""
    pending, started, done = list(gids), {}, []

    while pending or started:
        live = list_tasks(prefix, states={'READY','RUNNING'})
        while pending and len(live) < max_inflight:
            gid = pending.pop(0)
            name = f"{prefix}{gid}_{ANCHOR_YEAR}"
            try:
                started[gid] = export_table(build_fn(gid), name)
                print(f"[{datetime.now():%H:%M}] started  {gid}")
            except Exception as e:
                print(f"[{datetime.now():%H:%M}] SUBMIT FAILED {gid}: {e}")
            live = list_tasks(prefix, states={'READY','RUNNING'})

        for t in list_tasks(prefix, states={'FAILED'}):
            d = t['description']
            if d not in done:
                done.append(d)
                print(f"[{datetime.now():%H:%M}] FAILED   {d} -> {t.get('error_message')}")
        for t in list_tasks(prefix, states={'COMPLETED'}):
            d = t['description']
            if d not in done:
                done.append(d)
                print(f"[{datetime.now():%H:%M}] done     {d}")

        if not pending and not list_tasks(prefix, states={'READY','RUNNING'}):
            break
        time.sleep(poll_s)

    print('\nfinished')
    task_summary(prefix)


# --- full Analysis 1 DW10 leg, smallest first --------------------------------
# NOTE: 'GHA' is not in SUBSET; the filter below drops it. Add to SUBSET if wanted.
# ORDER = ['PRY','ZMB','PNG','MDG','MYS','UKR','THA','GHA','FRA','ETH','NGA',
#          'CHL','PER','ZAF','MEX','IDN','IND','CHN','USA','CAN','BRA','AUS']
# ORDER = [g for g in ORDER if g in SUBSET]
# submit_throttled(ORDER, build_dw10_auto, f'p2v2_DW10_')   # auto-splits ZMB/PNG

## 10 · Analysis 1 — remaining three products

CCI300 full/harmonized and DW300 all run at 300 m, where the annulus is only ~4 px in
radius. These are cheap relative to the DW10 leg and can go as single multi-country
tasks. Run these *after* the DW10 smoke test confirms the pipeline.

In [ ]:
regions_subset = ee.FeatureCollection(GADM_ASSET).filter(
    ee.Filter.inList(ID_FIELD, list(SUBSET)))


def build_300m_products(year=ANCHOR_YEAR):
    lc = (ee.ImageCollection(CCI_COLLECTION)
            .filterDate(f'{year}-01-01', f'{year+1}-01-01')
            .first().select(CCI_BAND))

    nat_full, ag_full = cci_integrity(lc, harmonized=False)
    nat_harm, ag_harm = cci_integrity(lc, harmonized=True)

    out = {
        'CCI300_full': process(nat_full, ag_full, INNER_RADIUS_M, OUTER_RADIUS_M,
                               SCALE_300, regions_subset, year, 'CCI300_full',
                               tile_scale=TILESCALE_300),
        'CCI300_harm': process(nat_harm, ag_harm, INNER_RADIUS_M, OUTER_RADIUS_M,
                               SCALE_300, regions_subset, year, 'CCI300_harm',
                               tile_scale=TILESCALE_300),
    }
    return out


def build_dw300(gid, year=ANCHOR_YEAR):
    """DW at 300 m, per country. Cheap kernel; still per-country to bound the composite."""
    fc  = country_fc(gid)
    label   = dw_annual_label(year, fc.geometry())
    nat, ag = dw_integrity(label)
    return process(nat, ag, INNER_RADIUS_M, OUTER_RADIUS_M, SCALE_300,
                   fc, year, 'DW300', tile_scale=TILESCALE_300)


# for k, fc in build_300m_products().items():
#     export_table(fc, f"p2v2_{k}_{ANCHOR_YEAR}")
# submit_throttled(SUBSET, build_dw300, 'p2v2_DW300_')

## 11 · Analyses 2 and 3 — unchanged stubs

Both inherit every fix above. Analysis 3 is the one to watch: at native 10 m with
`INNER_RADIUS_M_10M = 5`, the annulus is effectively a full 115 px disc, which is
strictly heavier than the Analysis 1 configuration. Expect to need `preaggregate_m`
or sub-national tiling there.

In [ ]:
# TODO(Analysis 2): spliced 300 m series 2000-2025
#   CCI_YEARS = range(2000, 2022);  DW_YEARS = range(2016, 2026);  OVERLAP = range(2016, 2022)
#   - loop CCI-300 over CCI_YEARS (Phase 1 tiling, full crosswalk)
#   - loop DW-300  over DW_YEARS  (dw_annual_label -> dw_integrity -> process @ 300 m)
#   - fit per-country offset  fi_DW ~ fi_CCI  on OVERLAP -> splice correction

# TODO(Analysis 3): native 10 m series 2016-2025
#   for yr in range(2016, 2026):
#       nat, ag = dw_integrity(dw_annual_label(yr, geo))
#       fc = process(nat, ag, INNER_RADIUS_M_10M, OUTER_RADIUS_M, SCALE_10,
#                    country_fc(gid), yr, f'DW10_{yr}',
#                    tile_scale=16, preaggregate_m=50)
#   NOTE: 2026 partial (year incomplete).
pass

## Notes

- **Service-shed held in metres**, so 300 m and 10 m measure the same physical catchment;
  pixel footprints differ by grain (~4 px radius @ 300 m, ~115 px @ 10 m).
- **Analysis 1 validates Analysis 2**: its `DW300` config == one slice of the 300 m series.
- **Legend term** isolated via `CCI300_harm` (mosaics collapsed to dominant class).
- **DW label** = argmax of mean probability, computed band-wise rather than via arrays.
- **Output field renamed** `gritM` -> `fi` (functional integrity), matching the FSCI
  indicator name. Update the Phase 1 loader's groupby columns accordingly.
- **Combine/dedup** exported CSVs with the Phase 1 loader
  (groupby-mean on `[GID_0, year, product, tile]`).
- If `preaggregate_m` is used for any country, record which ones — it is an
  approximation and belongs in the methods.
